# Notebook 03 — Feature Engineering

**Goal:** Create the **target** (what we predict) and **features** (what the model uses to predict).

**Pipeline step:** Feature engineering

**Input file:** `data/processed/matches_2021_onwards.csv` (from Notebook 02)

**Output file:** `data/processed/matches_ready_for_model.csv`

# Step 1 — Load the Cleaned Data

We start from the **processed** file — not raw data.
This file already has missing scores removed and only 2021+ matches.

In [1]:
import pandas as pd
from pathlib import Path

# Build paths
PROJECT_ROOT = Path("..")
INPUT_PATH = PROJECT_ROOT / "data" / "processed" / "matches_2021_onwards.csv"
OUTPUT_PATH = PROJECT_ROOT / "data" / "processed" / "matches_ready_for_model.csv"

# Load cleaned match data
df = pd.read_csv(INPUT_PATH)

print("Rows loaded:", len(df))
print("Columns:", list(df.columns))
df.head()

Rows loaded: 5711
Columns: ['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral']


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,2021-01-12,United Arab Emirates,Iraq,0.0,0.0,Friendly,Dubai,United Arab Emirates,False
1,2021-01-18,Kuwait,Palestine,0.0,1.0,Friendly,Kuwait City,Kuwait,False
2,2021-01-19,Dominican Republic,Puerto Rico,0.0,1.0,Friendly,Santo Domingo,Dominican Republic,False
3,2021-01-22,Guatemala,Puerto Rico,1.0,0.0,Friendly,Guatemala City,Guatemala,False
4,2021-01-25,Dominican Republic,Serbia,0.0,0.0,Friendly,Santo Domingo,Dominican Republic,False


# Step 2 — Create the Target Column

**Target** = the answer we want the model to predict.

For each match, home team result:
- **Win** — home_score > away_score
- **Draw** — home_score == away_score
- **Loss** — home_score < away_score

We use scores **only here** to create the label. The model will NOT see scores during training.

In [2]:
def get_home_result(row):
    """Returns Win, Draw, or Loss for the home team based on final scores."""
    if row["home_score"] > row["away_score"]:
        return "Win"
    elif row["home_score"] == row["away_score"]:
        return "Draw"
    else:
        return "Loss"


# apply() runs the function on every row; axis=1 means row-by-row
df["home_result"] = df.apply(get_home_result, axis=1)

# Check the target looks correct
print("Target value counts:")
print(df["home_result"].value_counts())

df[["home_team", "away_team", "home_score", "away_score", "home_result"]].head()

Target value counts:
home_result
Win     2741
Loss    1662
Draw    1308
Name: count, dtype: int64


,home_team,away_team,home_score,away_score,home_result
0,United Arab Emirates,Iraq,0.0,0.0,Draw
1,Kuwait,Palestine,0.0,1.0,Loss
2,Dominican Republic,Puerto Rico,0.0,1.0,Loss
3,Guatemala,Puerto Rico,1.0,0.0,Win
4,Dominican Republic,Serbia,0.0,0.0,Draw


# Step 3 — Data Leakage Warning

**Data leakage** = giving the model information it would not have before the match ends.

If we use `home_score` and `away_score` as features, the model "cheats" — it already knows the answer.

**Rule:** Features must be things known **before** the final whistle (teams, tournament, date, neutral venue).

We will **drop scores** from our modeling dataset before saving.

# Step 4 — Create New Features from Existing Columns

**Feature engineering** = creating useful input columns from raw data.

We create:
| New feature | From | Why |
|-------------|------|-----|
| `year` | date | Recent years may play differently |
| `month` | date | Season timing can matter |
| `is_neutral` | neutral | No home crowd advantage on neutral ground |

In [3]:
# Convert date text to datetime
df["date"] = pd.to_datetime(df["date"])

# Extract year and month as numbers
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month

# Convert True/False neutral flag to 1/0 (easier for ML models later)
df["is_neutral"] = df["neutral"].astype(int)

print("New features sample:")
df[["date", "year", "month", "neutral", "is_neutral"]].head()

New features sample:


,date,year,month,neutral,is_neutral
0,2021-01-12,2021,1,False,0
1,2021-01-18,2021,1,False,0
2,2021-01-19,2021,1,False,0
3,2021-01-22,2021,1,False,0
4,2021-01-25,2021,1,False,0


# Step 5 — Choose Feature Columns for the Model

These are the **inputs** our model will use:

- `home_team` — who plays at home
- `away_team` — opponent
- `tournament` — type of competition
- `is_neutral` — 1 if neutral venue, 0 if not
- `year` — match year
- `month` — match month

**Target column:** `home_result` (Win / Draw / Loss)

Text columns (`home_team`, `away_team`, `tournament`) will need **encoding** in the next notebook — models need numbers, not words.

In [4]:
# List of input feature column names
FEATURE_COLUMNS = [
    "home_team",
    "away_team",
    "tournament",
    "is_neutral",
    "year",
    "month",
]

# Target column name
TARGET_COLUMN = "home_result"

# Build final modeling table: features + target only (NO scores — no leakage)
model_df = df[FEATURE_COLUMNS + [TARGET_COLUMN]].copy()

print("Modeling dataset shape:", model_df.shape)
print("\nFirst 5 rows:")
model_df.head()

Modeling dataset shape: (5711, 7)

First 5 rows:


,home_team,away_team,tournament,is_neutral,year,month,home_result
0,United Arab Emirates,Iraq,Friendly,0,2021,1,Draw
1,Kuwait,Palestine,Friendly,0,2021,1,Loss
2,Dominican Republic,Puerto Rico,Friendly,0,2021,1,Loss
3,Guatemala,Puerto Rico,Friendly,0,2021,1,Win
4,Dominican Republic,Serbia,Friendly,0,2021,1,Draw


# Step 6 — Check Target Balance

If one class is much more common (e.g. 90% Win), the model may learn to always guess that class.

We check proportions so we know what to expect in evaluation later.

In [5]:
# value_counts(normalize=True) shows percentages
target_pct = model_df[TARGET_COLUMN].value_counts(normalize=True) * 100

print("Target distribution (%):")
print(target_pct.round(1))

print("\nUnique home teams:", model_df["home_team"].nunique())
print("Unique away teams:", model_df["away_team"].nunique())
print("Unique tournaments:", model_df["tournament"].nunique())

Target distribution (%):
home_result
Win     48.0
Loss    29.1
Draw    22.9
Name: proportion, dtype: float64

Unique home teams: 255
Unique away teams: 252
Unique tournaments: 66


# Step 7 — Save Modeling-Ready Data

Save to `data/processed/matches_ready_for_model.csv`.

Next notebook will: **train/test split → encode text → train first model**.

In [7]:
# Save the modeling dataset
model_df.to_csv(OUTPUT_PATH, index=False)

print("Saved to:", OUTPUT_PATH.resolve())
print("File exists:", OUTPUT_PATH.exists())
print("Rows saved:", len(model_df))
print("Columns saved:", list(model_df.columns))

Saved to: C:\Users\HP-15\Desktop\worldcup_predictor\data\processed\matches_ready_for_model.csv
File exists: True
Rows saved: 5711
Columns saved: ['home_team', 'away_team', 'tournament', 'is_neutral', 'year', 'month', 'home_result']


# Summary

| Item | Value |
|------|-------|
| Input | `matches_2021_onwards.csv` (5,711 matches) |
| Target | `home_result` → Win / Draw / Loss |
| Features | home_team, away_team, tournament, is_neutral, year, month |
| Removed | home_score, away_score (prevent data leakage) |
| Output | `matches_ready_for_model.csv` |

**Next notebook:** Train/test split, encoding, and train our first model.